# Modelo de Predicción de Churn — Opción A (Binario)

## Caso Práctico - Empresa de Telecomunicaciones
## MBI — Prácticas Aplicadas 2026

---

En este notebook construimos el primer modelo de predicción de churn.

Seguimos la **Opción A**: una fila por cliente con la etiqueta `ever_churn` (¿churneó en algún momento del período observado?). Es el enfoque más directo y nos permite validar rápidamente qué variables funcionan.

**El flujo que seguimos:**
1. Construir el dataset final integrando las 5 fuentes
2. Preparar las variables (encoding, imputación, selección)
3. Entrenar un modelo baseline (Regresión Logística)
4. Entrenar un segundo modelo (Random Forest)
5. Comparar resultados y extraer conclusiones
6. Interpretar el modelo: ¿qué variables explican el churn?


## Nota sobre leakage

Al usar `ever_churn` estamos agregando toda la historia del cliente. Esto es una simplificación válida para el EDA y para validar variables, pero tiene un riesgo: algunas variables (impagos, caída de consumo) pueden haber ocurrido justo en el mes del churn, lo que sería usar información del futuro.

Para este modelo lo aceptamos conscientemente, sabiendo que en el modelo temporal (Opción B) habrá que aplicar lags correctamente.


## 1. Librerías


In [ ]:
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    RocCurveDisplay, ConfusionMatrixDisplay
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
sns.set_theme(style='whitegrid', context='notebook')

ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

RANDOM_STATE = 42
print('Librerías cargadas')

---
## 2. Construcción del dataset final

Integramos las 5 fuentes de datos. Cada fila es un cliente y cada columna es una feature que resume su comportamiento a lo largo del período observado.

Las variables que incluimos son las que identificamos como relevantes en los EDAs:
- **Perfil del cliente**: edad, antigüedad, ingresos, tipo de plan, zona
- **Facturación**: importe medio, impagos, retrasos, stress de red, descuentos
- **Soporte**: número de interacciones, tasa de resolución, satisfacción, llamadas de baja
- **Calidad de red**: índice de calidad, cobertura 5G, tasa de cortes
- **Encuestas**: NPS, sentimiento, puntuación de la zona


In [ ]:
# Cargamos y limpiamos usando los módulos del proyecto
# load_all()  → lee los 6 CSV con fechas ya parseadas
# clean_all() → aplica toda la limpieza centralizada en clean.py
from src.load import load_all
from src.clean import clean_all

data  = load_all()
clean = clean_all(data, save=False)

clientes  = clean['clientes']
churn     = clean['churn']
factura   = clean['facturacion']
soporte   = clean['soporte']
calidad   = clean['calidad']
encuestas = clean['encuestas']

print('\nDatos cargados y limpios — listos para modelar')

In [ ]:
# ── Target ────────────────────────────────────────────────────────────────
churn_agg = churn.groupby('cliente_id').agg(
    ever_churn         = ('churn', 'max'),
    n_meses_observados = ('churn', 'count'),
).reset_index()

# ── Facturación ───────────────────────────────────────────────────────────
factura_agg = factura.groupby('cliente_id').agg(
    importe_medio       = ('importe_total',        'mean'),
    consumo_extra_medio = ('consumo_extra',         'mean'),
    pct_meses_impago    = ('impago_flag',            'mean'),
    dias_retraso_medio  = ('dias_retraso_pago',      'mean'),
    stress_medio        = ('stress_calidad_lag',     'mean'),
    pct_meses_descuento = ('descuento_aplicado',     lambda x: (x > 0).mean()),
    variacion_consumo   = ('variacion_consumo_pct',  'mean'),
    n_meses_facturados  = ('importe_total',          'count'),
).reset_index()

plan_dom = (factura.dropna(subset=['tipo_plan'])
            .groupby('cliente_id')['tipo_plan']
            .agg(lambda x: x.value_counts().index[0])
            .reset_index()
            .rename(columns={'tipo_plan': 'plan_factura'}))
factura_agg = factura_agg.merge(plan_dom, on='cliente_id', how='left')

# ── Soporte ───────────────────────────────────────────────────────────────
soporte_agg = soporte.groupby('cliente_id').agg(
    n_interacciones    = ('interaccion_id',    'count'),
    tasa_resolucion    = ('resuelto',          'mean'),
    satisfaccion_media = ('satisfaccion_post', 'mean'),
    n_impagos_sop      = ('impago_mes',        'sum'),
    n_mot_baja         = ('motivo', lambda x: (x == 'Baja / portabilidad').sum()),
).reset_index()

# ── Calidad de red por zona ───────────────────────────────────────────────
# Nota: clean_calidad() ya corrigió los errores tipográficos de tipo_zona
calidad_zona = calidad.groupby('zona_id').agg(
    calidad_global_media = ('indice_calidad_global', 'mean'),
    cobertura_5g_media   = ('cobertura_5g_pct',      'mean'),
    tasa_cortes_media    = ('tasa_cortes_pct',        'mean'),
).reset_index()

# ── Encuestas por zona ────────────────────────────────────────────────────
# Nota: clean_encuestas() ya trató los valores de NPS y puntuación fuera de rango
enc_zona = encuestas.groupby('zona_id').agg(
    nps_medio         = ('nps_0a10',              'mean'),
    sentimiento_medio = ('sent_text_latente',      'mean'),
    puntuacion_media  = ('puntuacion_general_1a5', 'mean'),
).reset_index()

# ── Join final ────────────────────────────────────────────────────────────
df = clientes.merge(churn_agg,   on='cliente_id', how='inner')
df = df.merge(factura_agg,       on='cliente_id', how='left')
df = df.merge(soporte_agg,       on='cliente_id', how='left')
df = df.merge(calidad_zona,      on='zona_id',    how='left')
df = df.merge(enc_zona,          on='zona_id',    how='left')

# Clientes sin soporte → 0 interacciones
for col in ['n_interacciones', 'n_impagos_sop', 'n_mot_baja']:
    df[col] = df[col].fillna(0)

print(f'Dataset final: {df.shape[0]:,} clientes x {df.shape[1]} columnas')
print(f'Ever churn=1:  {df["ever_churn"].sum():,} ({df["ever_churn"].mean()*100:.1f}%)')
print(f'Ever churn=0:  {(df["ever_churn"]==0).sum():,} ({(df["ever_churn"]==0).mean()*100:.1f}%)')

---
## 3. Preparación de variables

Antes de entrenar el modelo hay que preparar las variables. Distinguimos **tres grupos** según la naturaleza de cada variable:

1. **Variables numéricas continuas** (edad, ingresos, importe medio...) → `StandardScaler`: homogeneizamos las escalas para que variables con rangos grandes no dominen sobre las demás.

2. **Variables nominales** (región, sexo, tipo de dispositivo...) → `OneHotEncoder` con `drop='first'`: son categorías sin jerarquía. Si las tratáramos como números el modelo asumiría relaciones de orden que no existen. El `drop='first'` evita multicolinealidad.

3. **Variables ordinales** (tipo de plan: Básico < Estándar < Premium) → mapeo manual a números (1, 2, 3) + `StandardScaler`: sí tienen un orden real que queremos que el modelo capture.


In [ ]:
# Separamos las variables en tres grupos según su naturaleza

# 1. Numéricas continuas → StandardScaler
FEATURES_NUM = [
    # Perfil del cliente
    'edad', 'num_lineas', 'ingreso_estimado', 'antiguedad_meses',
    'descuento_activo', 'poblacion_zona',
    # Facturación
    'importe_medio', 'consumo_extra_medio', 'pct_meses_impago',
    'dias_retraso_medio', 'stress_medio', 'pct_meses_descuento',
    'variacion_consumo', 'n_meses_facturados',
    # Soporte
    'n_interacciones', 'tasa_resolucion', 'satisfaccion_media',
    'n_impagos_sop', 'n_mot_baja',
    # Calidad de red
    'calidad_global_media', 'cobertura_5g_media', 'tasa_cortes_media',
    # Encuestas
    'nps_medio', 'sentimiento_medio', 'puntuacion_media',
]

# 2. Nominales (sin orden) → OneHotEncoder
FEATURES_CAT_NOMINAL = [
    'region',           # Norte, Sur, Este, Oeste, Centro — sin jerarquía
    'tipo_zona',        # rural, suburbana, urbana_premium — sin jerarquía
    'sexo',             # Hombre, Mujer
    'estado_civil',     # Soltero/a, Casado/a, Divorciado/a
    'tipo_dispositivo', # distintos modelos de dispositivo
    'plan_factura',     # Prepago, Contrato, Premium (desde facturación)
]

# 3. Ordinales (con orden) → mapeo manual + StandardScaler
# tipo_plan: Básico < Estándar < Premium — sí tiene jerarquía
MAPA_TIPO_PLAN = {'Básico': 1, 'Estándar': 2, 'Premium': 3}
df['tipo_plan_num'] = df['tipo_plan'].map(MAPA_TIPO_PLAN)
# Los nulos (planes desconocidos) los imputamos con la mediana después
FEATURES_ORDINAL = ['tipo_plan_num']

TARGET = 'ever_churn'

# Verificar que todas las columnas existen
all_features = FEATURES_NUM + FEATURES_CAT_NOMINAL + FEATURES_ORDINAL
missing = [f for f in all_features if f not in df.columns]
if missing:
    print(f'⚠️ Columnas no encontradas: {missing}')
else:
    print('✅ Todas las variables disponibles')
    print(f'   Numéricas continuas: {len(FEATURES_NUM)}')
    print(f'   Nominales:           {len(FEATURES_CAT_NOMINAL)}')
    print(f'   Ordinales:           {len(FEATURES_ORDINAL)}')
    print(f'   Total:               {len(all_features)}')

In [ ]:
# Separamos X e y con los tres grupos de variables
X = df[FEATURES_NUM + FEATURES_CAT_NOMINAL + FEATURES_ORDINAL].copy()
y = df[TARGET].copy()

# Train / Test split — estratificado para mantener la proporción de churn
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train.shape[0]:,} clientes ({y_train.mean()*100:.1f}% churn)')
print(f'Test:  {X_test.shape[0]:,} clientes  ({y_test.mean()*100:.1f}% churn)')
print()
print('Nulos en X_train:')
nulos = X_train.isnull().sum()
print(nulos[nulos > 0])

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Pipeline para variables numéricas continuas
# Imputamos con la mediana (más robusta que la media ante outliers)
# y escalamos para homogeneizar rangos
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

# Pipeline para variables nominales (sin orden)
# Imputamos con 'desconocido' y aplicamos OneHotEncoder
# drop='first' elimina una columna por variable para evitar multicolinealidad
nom_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='desconocido')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)),
])

# Pipeline para variables ordinales (con orden: Básico < Estándar < Premium)
# Ya están mapeadas a números (1, 2, 3), solo imputamos y escalamos
ord_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

# Combinamos los tres pipelines en un ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_pipeline, FEATURES_NUM),
    ('nom', nom_pipeline, FEATURES_CAT_NOMINAL),
    ('ord', ord_pipeline, FEATURES_ORDINAL),
])

print('Preprocesador definido con 3 pipelines:')
print('  num → imputar mediana + StandardScaler   (variables numéricas continuas)')
print('  nom → imputar desconocido + OneHotEncoder (variables nominales sin orden)')
print('  ord → imputar mediana + StandardScaler   (variables ordinales con orden)')

---
## 4. Modelo 1 — Regresión Logística (Baseline)

Empezamos con el modelo más sencillo posible. La regresión logística es un buen punto de partida porque:
- Es muy interpretable: los coeficientes nos dicen directamente qué variables suben o bajan la probabilidad de churn
- Es rápida de entrenar
- Nos da un AUC de referencia para comparar con modelos más complejos

Si un modelo complejo no mejora mucho a la regresión logística, probablemente no valga la complejidad añadida.


In [ ]:
# Definimos el pipeline completo: preprocesado + modelo
pipe_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(random_state=RANDOM_STATE, max_iter=1000, C=1.0))
])

# Validación cruzada con 5 folds estratificados
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
auc_cv_lr = cross_val_score(pipe_lr, X_train, y_train, cv=cv, scoring='roc_auc')

print(f'Regresión Logística — AUC en CV (5 folds):')
print(f'  Por fold: {[round(x,3) for x in auc_cv_lr]}')
print(f'  Media:    {auc_cv_lr.mean():.3f} ± {auc_cv_lr.std():.3f}')

# Entrenamos sobre todo el train y evaluamos en test
pipe_lr.fit(X_train, y_train)
y_pred_proba_lr = pipe_lr.predict_proba(X_test)[:, 1]
auc_test_lr = roc_auc_score(y_test, y_pred_proba_lr)
print(f'\n  AUC en test: {auc_test_lr:.3f}')

In [ ]:
# Interpretación: coeficientes de la regresión logística
# Reconstruimos los nombres de todas las variables tras el preprocesado

# Nombres de las variables nominales tras el OneHotEncoder
nom_feature_names = (pipe_lr.named_steps['preprocessor']
                     .named_transformers_['nom']
                     .named_steps['encoder']
                     .get_feature_names_out(FEATURES_CAT_NOMINAL))

# Combinamos: numéricas + nominales (one-hot) + ordinales
feature_names = FEATURES_NUM + list(nom_feature_names) + FEATURES_ORDINAL

coefs = pd.Series(
    pipe_lr.named_steps['model'].coef_[0],
    index=feature_names
).sort_values(key=abs, ascending=False)

# Top 20 variables más influyentes
top20 = coefs.head(20).sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#E85C4C' if v > 0 else '#4C9BE8' for v in top20.values]
ax.barh(top20.index, top20.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Regresión Logística — Top 20 variables más influyentes\n'
             '(rojo = aumenta la probabilidad de churn | azul = la reduce)',
             fontweight='bold')
ax.set_xlabel('Coeficiente (escala estandarizada)')
plt.tight_layout()
plt.show()

### Resultado — Regresión Logística

**AUC en validación cruzada: 0.981 ± 0.005 | AUC en test: 0.991**

Es un resultado muy alto. El modelo logístico, siendo el más simple de los dos, ya es capaz de separar churners de no churners con una precisión excelente. La estabilidad entre folds (0.974 - 0.986) confirma que no es un resultado por azar.

**Interpretación de los coeficientes:**

Las variables que más aumentan la probabilidad de churn (barras rojas):
- `stress_medio` → es la señal más fuerte con diferencia. Un cliente en una zona con alta degradación de red tiene mucha más probabilidad de abandonar.
- `cobertura_5g_media` y `calidad_global_media` → también de red. Zonas con peor cobertura 5G y peor calidad global generan más churn.
- `region_Este`, `tipo_zona_suburbana`, `plan_factura_Prepago` → diferencias geográficas y de tipo de plan que el modelo captura.
- `pct_meses_impago` → los impagos frecuentes son señal de riesgo.

Las variables que más reducen la probabilidad de churn (barras azules):
- `n_meses_facturados` → es el coeficiente más grande en valor absoluto. Cuantos más meses lleva un cliente facturando, menos probabilidad tiene de irse. Confirma lo que vimos en el EDA: la antigüedad fideliza.
- `n_interacciones` → más contactos con soporte se asocian a menos churn. Puede indicar que clientes más activos están más comprometidos con el servicio.
- `estado_civil_desconocido` → clientes con estado civil no registrado tienen menos churn, probablemente por ser un segmento específico.

⚠️ `n_meses_facturados` tiene un coeficiente muy alto (-8). Esto puede ser parcialmente un efecto de leakage: los clientes que churnearon llevan menos meses porque se fueron antes. En el modelo temporal (Opción B) esto se trata correctamente.


---
## 5. Modelo 2 — Random Forest

El Random Forest es un modelo más potente que puede capturar relaciones no lineales entre variables. Es menos interpretable que la regresión logística, pero podemos usar la importancia de variables para entender qué está usando.

Lo comparamos con el baseline para ver si la complejidad añadida vale la pena.


In [ ]:
pipe_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=20,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

auc_cv_rf = cross_val_score(pipe_rf, X_train, y_train, cv=cv, scoring='roc_auc')

print(f'Random Forest — AUC en CV (5 folds):')
print(f'  Por fold: {[round(x,3) for x in auc_cv_rf]}')
print(f'  Media:    {auc_cv_rf.mean():.3f} ± {auc_cv_rf.std():.3f}')

pipe_rf.fit(X_train, y_train)
y_pred_proba_rf = pipe_rf.predict_proba(X_test)[:, 1]
auc_test_rf = roc_auc_score(y_test, y_pred_proba_rf)
print(f'\n  AUC en test: {auc_test_rf:.3f}')

In [ ]:
# Importancia de variables del Random Forest
importancias = pd.Series(
    pipe_rf.named_steps['model'].feature_importances_,
    index=feature_names
).sort_values(ascending=False)

top15 = importancias.head(15).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
top15.plot(kind='barh', ax=ax, color='steelblue', alpha=0.85)
ax.set_title('Random Forest — Top 15 variables más importantes',
             fontweight='bold')
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

print('Top 10 variables:')
print(importancias.head(10).round(4).to_string())

### Resultado — Random Forest

**AUC en validación cruzada: 0.980 ± 0.005 | AUC en test: 0.994**

El Random Forest mejora ligeramente al modelo logístico en test (0.994 vs 0.991), pero la diferencia es mínima. Ambos modelos tienen un rendimiento excelente.

**Importancia de variables:**

El árbol confirma y matiza lo que vimos en la regresión logística:
- `n_meses_facturados` acapara el 45% de la importancia total. Es la variable dominante.
- `stress_medio` (16.6%) y `n_interacciones` (16.4%) son las siguientes más importantes. El estrés de red y el número de contactos con soporte son las señales de comportamiento más relevantes.
- `variacion_consumo` (4.3%) y `pct_meses_impago` (3.8%) confirman que los cambios en el comportamiento económico del cliente también predicen el abandono.
- `n_mot_baja` aparece con un 2.4% — las llamadas explícitas para darse de baja son señal directa aunque no son el predictor más importante (muchos clientes se van sin llamar).

Las variables de encuestas (NPS, sentimiento) tienen poca importancia individual en el Random Forest porque su información ya está capturada por el `stress_medio`, con el que están altamente correlacionadas.


---
## 6. Comparación de modelos


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curvas ROC
RocCurveDisplay.from_predictions(
    y_test, y_pred_proba_lr,
    name=f'Regresión Logística (AUC={auc_test_lr:.3f})',
    color='steelblue', ax=axes[0]
)
RocCurveDisplay.from_predictions(
    y_test, y_pred_proba_rf,
    name=f'Random Forest (AUC={auc_test_rf:.3f})',
    color='#E85C4C', ax=axes[0]
)
axes[0].plot([0,1],[0,1], 'k--', label='Baseline (AUC=0.500)')
axes[0].set_title('Curva ROC — Comparación de modelos', fontweight='bold')
axes[0].legend()

# Comparativa de AUC
resultados = pd.DataFrame({
    'Modelo': ['Regresión Logística', 'Random Forest'],
    'AUC CV (media)': [auc_cv_lr.mean(), auc_cv_rf.mean()],
    'AUC Test': [auc_test_lr, auc_test_rf],
})

x = range(len(resultados))
bars1 = axes[1].bar([i - 0.2 for i in x], resultados['AUC CV (media)'],
                    width=0.35, label='AUC CV', color='steelblue', alpha=0.8)
bars2 = axes[1].bar([i + 0.2 for i in x], resultados['AUC Test'],
                    width=0.35, label='AUC Test', color='#E85C4C', alpha=0.8)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(resultados['Modelo'])
axes[1].set_ylim(0.5, 1.0)
axes[1].set_title('Comparativa AUC: CV vs Test', fontweight='bold')
axes[1].set_ylabel('AUC-ROC')
axes[1].legend()
axes[1].axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
for bar in list(bars1) + list(bars2):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print('\nResumen:')
display(resultados.round(3))

---
## 7. Matriz de confusión y umbral de decisión

El AUC mide qué tan bien separa el modelo churners de no churners. Pero para tomar decisiones de negocio necesitamos elegir un **umbral**: a partir de qué probabilidad predicha consideramos que un cliente va a churnearse.

Por defecto los modelos usan 0.5, pero en churn suele convenir bajarlo porque:
- Retener un cliente que no iba a irse cuesta poco (un descuento)
- Perder un cliente que sí iba a irse cuesta mucho (ingresos perdidos)

Usamos el mejor modelo de los dos para este análisis.


In [ ]:
# Usamos el mejor modelo
mejor_modelo = pipe_rf if auc_test_rf >= auc_test_lr else pipe_lr
nombre_mejor = 'Random Forest' if auc_test_rf >= auc_test_lr else 'Regresión Logística'
y_pred_proba_best = pipe_rf.predict_proba(X_test)[:, 1] if auc_test_rf >= auc_test_lr else y_pred_proba_lr

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Umbral 0.5 (defecto)
y_pred_05 = (y_pred_proba_best >= 0.5).astype(int)
cm_05 = confusion_matrix(y_test, y_pred_05)
ConfusionMatrixDisplay(cm_05, display_labels=['No Churn', 'Churn']).plot(
    ax=axes[0], colorbar=False, cmap='Blues'
)
axes[0].set_title(f'{nombre_mejor} — Umbral 0.5', fontweight='bold')

# Umbral 0.3 (más agresivo en detección)
y_pred_03 = (y_pred_proba_best >= 0.3).astype(int)
cm_03 = confusion_matrix(y_test, y_pred_03)
ConfusionMatrixDisplay(cm_03, display_labels=['No Churn', 'Churn']).plot(
    ax=axes[1], colorbar=False, cmap='Oranges'
)
axes[1].set_title(f'{nombre_mejor} — Umbral 0.3 (más recall)', fontweight='bold')

plt.suptitle('Matriz de confusión: efecto del umbral de decisión',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'=== Umbral 0.5 ===')
print(classification_report(y_test, y_pred_05, target_names=['No Churn', 'Churn']))
print(f'=== Umbral 0.3 ===')
print(classification_report(y_test, y_pred_03, target_names=['No Churn', 'Churn']))

---
## 8. Valor de negocio del modelo

Un AUC alto está bien, pero lo que le importa al negocio es cuánto dinero ahorra el modelo. Hacemos una estimación sencilla.

**Supuestos:**
- Ingreso medio mensual por cliente: ~128€ (del EDA de facturación)
- Vida media de un cliente retenido: 12 meses adicionales estimados
- Coste de retención (descuento/campaña): 30€ por cliente contactado
- Tasa de éxito de la campaña de retención: 30% de los clientes contactados se quedan


In [ ]:
# Parámetros de negocio
INGRESO_MENSUAL   = 128   # € por cliente
MESES_RETENIDO    = 12    # meses adicionales estimados si se retiene
COSTE_RETENCION   = 30    # € por cliente contactado (descuento + gestión)
TASA_EXITO        = 0.30  # 30% de los contactados se quedan

valor_cliente     = INGRESO_MENSUAL * MESES_RETENIDO  # €/cliente retenido

# Con umbral 0.3 (más recall)
tn, fp, fn, tp = cm_03.ravel()

clientes_detectados    = tp + fp   # a cuántos contactamos
clientes_retenidos     = int(tp * TASA_EXITO)  # de los verdaderos churners detectados
valor_generado         = clientes_retenidos * valor_cliente
coste_campana          = clientes_detectados * COSTE_RETENCION
beneficio_neto         = valor_generado - coste_campana
clientes_perdidos      = fn   # churners que no detectamos
valor_perdido          = clientes_perdidos * valor_cliente

print('=' * 50)
print('  ESTIMACIÓN DE VALOR DE NEGOCIO')
print('=' * 50)
print(f'  Clientes en test:           {len(y_test):,}')
print(f'  Churners reales en test:    {int(y_test.sum()):,}')
print()
print(f'  Churners detectados (TP):   {tp:,}')
print(f'  Falsos positivos (FP):      {fp:,}  ← contactamos pero no iban a irse')
print(f'  Churners perdidos (FN):     {fn:,}  ← no detectamos')
print()
print(f'  Clientes que contactamos:   {clientes_detectados:,}')
print(f'  Clientes retenidos (30%):   {clientes_retenidos:,}')
print()
print(f'  Valor generado:             {valor_generado:,} €')
print(f'  Coste campaña:              {coste_campana:,} €')
print(f'  Beneficio neto estimado:    {beneficio_neto:,} €')
print()
print(f'  Valor perdido (FN):         {valor_perdido:,} €')
print('=' * 50)

# Comparación: sin modelo vs con modelo
churners_totales = int(y_test.sum())
print(f'\n  Sin modelo (contactar a todos):')
coste_sin_modelo = len(y_test) * COSTE_RETENCION
retenidos_sin_modelo = int(churners_totales * TASA_EXITO)
beneficio_sin_modelo = retenidos_sin_modelo * valor_cliente - coste_sin_modelo
print(f'    Coste: {coste_sin_modelo:,} €  |  Retenidos: {retenidos_sin_modelo}  |  Beneficio: {beneficio_sin_modelo:,} €')
print(f'  Con modelo (umbral 0.3):')
print(f'    Coste: {coste_campana:,} €  |  Retenidos: {clientes_retenidos}  |  Beneficio: {beneficio_neto:,} €')

### Cómo leer este análisis

Los números son estimaciones con supuestos simplificados. El objetivo es mostrar que el modelo tiene **valor económico concreto**, no solo un AUC abstracto que es difícil de trasladar al negocio.

El mensaje clave no es 'el modelo tiene AUC 0.994', sino: **el modelo permite ahorrar 36.222€ en costes de campaña frente a contactar a todos los clientes, manteniendo prácticamente el mismo número de retenciones**.


---
## 9. Conclusiones

### Resultados de los modelos

| Modelo | AUC CV | AUC Test |
|--------|--------|----------|
| Regresión Logística | 0.981 | 0.991 |
| Random Forest | 0.980 | 0.994 |

Ambos modelos tienen un rendimiento excelente y muy similar. La diferencia de 0.003 en AUC no justifica por sí sola elegir el Random Forest sobre la Regresión Logística. En un entorno de negocio real, la Regresión Logística sería preferible por su mayor interpretabilidad: los coeficientes explican directamente qué variables suben o bajan el riesgo de cada cliente.

### Variables más importantes

Las tres variables con mayor poder predictivo en ambos modelos son:
1. **`n_meses_facturados`** — proxy de antigüedad del cliente. Cuanto menos tiempo lleva facturando, más riesgo de abandono. ⚠️ Variable sensible a leakage en el modelo binario.
2. **`stress_medio`** — estrés de calidad de la red. Confirma que la experiencia técnica del servicio es el principal driver del churn.
3. **`n_interacciones`** — número de contactos con soporte. Más interacciones se asocian a menos churn, posiblemente porque refleja un cliente más comprometido.

### Valor de negocio (umbral 0.3)

Con el modelo aplicado sobre el conjunto de test (2.000 clientes, 400 churners reales):

- Detectamos **376 de los 400 churners reales** (solo perdemos 24)
- Solo contactamos a **383 clientes** (376 TP + 7 FP)
- **Beneficio neto estimado: 160.542€** vs 124.320€ sin modelo
- **Ahorro en costes de campaña: 48.510€** (60.000€ sin modelo vs 11.490€ con modelo)

### Limitación principal y próximo paso

El modelo binario agrega toda la historia del cliente en una sola etiqueta. Esto tiene el riesgo de leakage en variables como `n_meses_facturados`. El modelo temporal (Opción B, 1 fila por cliente-mes con lags) elimina este problema y permite además predecir el riesgo mes a mes, lo que es más útil operativamente para el negocio.

---
*Predicción de Churn — Empresa de Telecomunicaciones | Prácticas Aplicadas 2026*
